# Agriculature Water Use estimation from observed ET

In [ ]:
import pathlib as pl
import xarray as xr
import pywatershed as pws
import warnings 

import jupyter_black

jupyter_black.load()

warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow")

In [ ]:
run_dir_exist_fail = False
domain_dir_spinup = pl.Path("../test_data/ucb_ag_spinup_2yr/")
domain_dir_analysis = pl.Path("../test_data/ucb_ag_analysis_2yr/")
notebook_dir = pl.Path("./ag_runs")
if not notebook_dir.exists():
    notebook_dir.mkdir()

## Experimental design

First 2 years:  
    1. run NHM config  
    2. run the wu spin up  
    
Second 2 years, restarts:  
    3. continue the NHM config  
    4. continue wu open-loop / spin up  
    5. run the wu analysis restarting from the spinup  




In [ ]:
nhm_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoff,
    pws.PRMSSoilzone,
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

wu_ol_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoffAg,
    pws.PRMSSoilzoneAg,
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

wu_analysis_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoffAg,
    pws.PRMSSoilzoneAgObsET,
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

In [ ]:
control_spinup_file = domain_dir_spinup / "nhm_ic_w_output_subset.control"
control_analysis_file = (
    domain_dir_analysis
    / "nhm_dynamic_2000_2020_w_output_subset_no_restart.control"
)

control_nhm_1 = pws.Control.load_prms(control_spinup_file)
control_ol_1 = pws.Control.load_prms(control_spinup_file)
control_nhm_2 = pws.Control.load_prms(control_analysis_file)
control_ol_2 = pws.Control.load_prms(control_analysis_file)
control_analysis = pws.Control.load_prms(control_analysis_file)

control_nhm_1.options["input_dir"] = domain_dir_spinup
control_ol_1.options["input_dir"] = domain_dir_spinup
control_nhm_2.options["input_dir"] = domain_dir_analysis
control_ol_2.options["input_dir"] = domain_dir_analysis
control_analysis.options["input_dir"] = domain_dir_analysis

# For the NHM configuration, this dosent matter but for the ol
# using PRMSSoilzoneAg (and not PRMSSoilzoneAgObsET) it does matter
control_nhm_2.options["iter_aet_flag"] = False
control_ol_2.options["iter_aet_flag"] = False

control_nhm_1.options["netcdf_output_dir"] = notebook_dir / "run_nhm_1"
control_ol_1.options["netcdf_output_dir"] = notebook_dir / "run_ol_1"
control_nhm_2.options["netcdf_output_dir"] = notebook_dir / "run_nhm_2"
control_ol_2.options["netcdf_output_dir"] = notebook_dir / "run_ol_2"
control_analysis.options["netcdf_output_dir"] = notebook_dir / "run_analysis"

control_nhm_1.options["restart_write"] = control_nhm_1.options[
    "netcdf_output_dir"
]
control_ol_1.options["restart_write"] = control_ol_1.options[
    "netcdf_output_dir"
]
control_nhm_2.options["restart_read"] = control_nhm_1.options[
    "netcdf_output_dir"
]
control_ol_2.options["restart_read"] = control_ol_1.options[
    "netcdf_output_dir"
]
control_analysis.options["restart_read"] = control_ol_1.options[
    "netcdf_output_dir"
]

control_nhm_1.options["netcdf_output_var_names"] = [
    "hru_ppt",
    "hru_actet",
    "perv_actet",
    "potet",
    "soil_moist",
    "soil_rechr",
]
control_ol_1.options["netcdf_output_var_names"] = [
    "hru_ppt",
    "hru_actet",
    "perv_actet",
    "hru_ag_actet",
    "potet",
    "soil_moist",
    "ag_soil_moist",
    "ag_soil_moist_prev",
    "soil_rechr",
    "ag_soil_rechr",
    "ag_irrigation_add",
]
# need to add some more?
control_nhm_2.options["netcdf_output_var_names"] = control_nhm_1.options[
    "netcdf_output_var_names"
]
control_ol_2.options["netcdf_output_var_names"] = control_ol_1.options[
    "netcdf_output_var_names"
]
control_analysis.options["netcdf_output_var_names"] = control_ol_1.options[
    "netcdf_output_var_names"
]

# document what this is about and when we need it.
control_nhm_1.options["intcp_changeover_in_net_rain"] = False
control_ol_1.options["intcp_changeover_in_net_rain"] = False
control_nhm_2.options["intcp_changeover_in_net_rain"] = False
control_ol_2.options["intcp_changeover_in_net_rain"] = False
control_analysis.options["intcp_changeover_in_net_rain"] = False

In [ ]:
def mk_run_dirs(control_list: list, exist_fail=True) -> bool:
    for cc in control_list:
        if (thedir := cc.options["netcdf_output_dir"]).exists():
            if exist_fail:
                raise ValueError(
                    f"{thedir} directory can not exist before running this notebook."
                )
            else:
                print(f"Run directory {thedir} exists, skipping")
                return False
        else:
            thedir.mkdir()
            return True

## ModelGraph dead end

In [ ]:
# model_nhm = pws.Model(nhm_processes, control=control_nhm, parameters=params_nhm)
# nhm_palette = pws.analysis.utils.colorbrewer.nhm_process_colors(model_nhm)
# pws.analysis.utils.colorbrewer.jupyter_palette(nhm_palette)
# pws.analysis.ModelGraph(
#         model_nhm,
#         hide_variables=False,
#         process_colors=nhm_palette,
#         # show_params=show_params,
#     ).SVG(verbose=True, dpi=48)

In [ ]:
# There is some issue with resolving some graphics
# model_wu_spinup = pws.Model(wu_spinup_processes, control=control_spinup, parameters=params_spinup, find_input_files=False)
# wu_spinup_palette = pws.analysis.utils.colorbrewer.nhm_process_colors(model_wu_spinup)
# pws.analysis.utils.colorbrewer.jupyter_palette(wu_spinup_palette)
# pws.analysis.ModelGraph(
#         model_wu_spinup,
#         hide_variables=True,
#         process_colors=wu_spinup_palette,
#         # show_params=show_params,
#     ).SVG(verbose=True, dpi=48)

## NHM run

In [ ]:
if mk_run_dirs(
    [
        control_nhm_1,
    ],
    exist_fail=run_dir_exist_fail,
):
    params_nhm_1 = pws.parameters.PrmsParameters.load(
        domain_dir_spinup / control_nhm_1.options["parameter_file"]
    )
    model_nhm_1 = pws.Model(
        nhm_processes, control=control_nhm_1, parameters=params_nhm_1
    )
    model_nhm_1.run(finalize=True)

In [ ]:
if mk_run_dirs(
    [
        control_nhm_2,
    ],
    exist_fail=run_dir_exist_fail,
):
    params_nhm_2 = pws.parameters.PrmsParameters.load(
        domain_dir_analysis / control_nhm_2.options["parameter_file"]
    )
    model_nhm_2 = pws.Model(
        nhm_processes, control=control_nhm_2, parameters=params_nhm_2
    )
    model_nhm_2.run(finalize=True)

## WU spinup and open loop

The PRMSRunoffAg, PRMSSoilzoneAg, and PRMSSoilzoneAgObsET have a an "ag_frac" input. This input is required by pywatershed in either case, if the ag frac is dynamic or static, where as in PRMS a default ag_frac would be found in the parameter file is a dynamic one was not requested. In PRMS-legacy mode, we must create this input file as a netcdf file (is that true, cant we pass the dynamic parameter file?). If instantiating our Model the pywatershed way, we have additional options. Following a PRMS-legacy instantiation, we can easily re-use an existing input files with xarray to define the a time-varying ag_frac input. 

/* Somewhere document the pws-way */

In [ ]:
if mk_run_dirs(
    [
        control_ol_1,
    ],
    exist_fail=run_dir_exist_fail,
):
    params_spinup_file = control_spinup_file.parent / control_ol_1.options.get(
        "parameter_file"
    )
    params_nhm = pws.parameters.PrmsParameters.load(params_spinup_file)
    params_spinup = pws.parameters.PrmsParameters.load(params_spinup_file)
    ag_frac = xr.load_dataarray(domain_dir_spinup / "tmin.nc")
    ag_frac[:, :] = params_spinup.parameters["ag_frac"]
    ag_frac.name = "ag_frac"
    ag_frac_file = domain_dir_spinup / "ag_frac.nc"
    if not ag_frac_file.exists():
        # I probably should NOT write this file here. SHOULD DELETE IT
        ag_frac.to_netcdf(ag_frac_file)
    params_ol_1 = pws.parameters.PrmsParameters.load(
        control_spinup_file.parent / control_ol_1.options.get("parameter_file")
    )

    model_ol_1 = pws.Model(
        wu_ol_processes, control=control_ol_1, parameters=params_ol_1
    )
    model_ol_1.run(finalize=True)

In [ ]:
if mk_run_dirs(
    [
        control_ol_2,
    ],
    exist_fail=run_dir_exist_fail,
):
    ag_frac_dyn = pws.utils.PrmsDynamicParameter.load(
        domain_dir_analysis / "dyn_ag_frac.param", control=control_ol_2
    )
    ag_frac_dyn_ds = ag_frac_dyn.daily_data_array
    ag_frac_dyn_ds.name = "ag_frac"
    ag_frac_dyn_nc_file = domain_dir_analysis / "ag_frac.nc"
    if not ag_frac_dyn_nc_file.exists():
        # I probably should NOT write this file here. SHOULD DELETE IT
        ag_frac_dyn_ds.to_netcdf(domain_dir_analysis / "ag_frac.nc")
    params_ol_2 = pws.parameters.PrmsParameters.load(
        control_analysis_file.parent
        / control_ol_2.options.get("parameter_file")
    )

    model_ol_2 = pws.Model(
        wu_ol_processes, control=control_ol_2, parameters=params_ol_2
    )
    model_ol_2.run(finalize=True)

## Agricultural water use analysis

In [ ]:
if mk_run_dirs(
    [
        control_analysis,
    ],
    exist_fail=run_dir_exist_fail,
):
    params_analysis = pws.parameters.PrmsParameters.load(
        control_analysis_file.parent
        / control_analysis.options.get("parameter_file")
    )

    model_analysis = pws.Model(
        wu_analysis_processes,
        control=control_analysis,
        parameters=params_analysis,
    )
    model_analysis.run(finalize=True)

## Comparisons

In [ ]:
ucb_shp = "../../data/pywatershed/pywatershed_gis/ucb_2yr/HRU_subset.shp"
run_colors = {
    "NHM": "#8da0cb",
    "OL": "#66c2a5",
    "Analysis": "#fc8d62",
}
comparer = pws.analysis.HRUComparisonPanel(
    shapefile_path=ucb_shp,
    variable_names=control_analysis.options["netcdf_output_var_names"]
    + ["ag_frac", "aet_observed", "pet_observed"],
    run_directories={
        "NHM": control_nhm_2.options["netcdf_output_dir"],
        "OL": control_ol_2.options["netcdf_output_dir"],
        "Analysis": control_analysis.options["netcdf_output_dir"],
    },
    input_directories={
        "NHM": control_nhm_2.options["input_dir"],  #  domain_dir_analysis
        "OL": control_ol_2.options["input_dir"],  # domain_dir_analysis
        "Analysis": control_analysis.options[
            "input_dir"
        ],  # domain_dir_analysis
    },
    # simplify_tolerance=500,  # detail of HRU polygons dialed w/ this parameter
)
app = comparer.create_app()
# Starting a separate panel view is more stable than trying to render in jupyter
app.show()  # start a separte panel viewer

The above code spawns an HRUComparisonPanel object in a new browser tab or window. For the requested variable names, the viewer allows by-HRU comparison of variables. It provides a spatial map, which shows either the temporal statistic of a single run or the difference between temporal statistics for any two selected runs. When an HRU is selected, it provides timeseries of the current variable for all runs in a single plot below. The panel let's us quickly get an idea of what we are looking at because it has many options and quick access to all the runs. Once we use it it to get oriented, we'll see next that we can use the HRUComparisonPanel object to create even more customized plots. First, we'll look at several screenshots of a potential exploration that can be followed with that viewer. 

To focus on "water use" or "agricultural irrigation" right off the bat, we will select the variable of interest for this purpose "ag_irrigation_add". First, in the panel results below, note the distribution of irrigated areas across this UCB domain. We will zoom in on several of these to get acquainted with the results. 

![Farmington, NM, panel](static/farmington_panel_ag_irrigation_add.png)

### HRU 84966 - Farmington, NM
Let's first focus on one of the HRUs with a value towards the higher end of the range found across the map. This HRU, called out on the map, has nhm_id 84966 and is located just south of Farmington, NM. In the timeseries plot above, we can see the repeating pattern that from about late May into October, irrigation is applied. We also see some anomalous large irrigation applications at the beginning of the timeseries, perhaps soil moisture defecits from lack of irrigation being met??

We'll zoom in to see its location more clearly. And we'll take a satellite view of the location to get a sense of if there is indeed agriculture present.


| Location View | Satellite View |
|:---:|:---:|
| ![Farmington, NM, political](static/farmington_map_political.png) | ![Farmington, NM, satellite](static/farmington_map_satellite.png) |

By selecting different variables, we can get some sense of the hydrology and agriculture in this HRU. We can individually look at all the variables shown in the following plots. 


| |
|:---:|
| ![](static/farmington_ag_irrigation_add.png) |
| ![](static/farmington_potet.png) |
| ![](static/farmington_pet_observed.png) |
| ![](static/farmington_aet_observed.png) |
| ![](static/farmington_hru_ppt.png) |
| ![](static/farmington_hru_actet.png) |
| ![](static/farmington_hru_ag_actet.png) |
| ![](static/farmington_perv_actet.png) |
| ![](static/farmington_ag_soil_moist.png) |
| ![](static/farmington_ag_soil_moist_prev.png) |


We see quickly that this is a water-limited HRU (and region) and that actual ET in the model is governed by the amount of precipitation available. The modeled potential ET appears a decent match to the "observed" (OpenET) values, but the actual ET is very different. The comparison of the actual ET from the 3 runs shows that while including an agriculture soilzone in the OL run helps mitigate some of the extreme "flashy" response of actual ET to rainfaill seen in the NHM run, the missing process of irrigation is essential to correctly modeling the observed actual ET in this region. The elevated observed actual ET from roughly April through October, in boths years, is a signatures of agriculture in this HRU and region. We can see that estimating agricultural irrigation use from the observations helps simulate actual ET curves which are more realistic. Using these observations in the model also appears to help with overestimating actual ET peaks outside the growing/irrigation season. However, these quantities are in separate plots. In a moment, we'll use additional features of the HRUComparisonPanel object to make clear comparisons of these various quantities. 

Another notable feature of the plots is that two large initial irrigation pulses are applied at the start of the run in January, which is unusual timing. This is the system response to the use of AET observations in the analysis. The OL and the Analysis run have identical states at the start of the plotted period (2000-2001), but we can see that their soil moisture levels diverge sharply as a result of needing to match observed AET. If we plot ag_soil_moist_prev, we see that the divergence is immediate. 

Let us make cleaner comparisons of modeled and observed quantities. Let's start with modeled and observed potential ET. 

In [ ]:
comparer.selected_hru_widget.value = 84966

# Plot different variables from different runs
plot_potet = comparer.hru_plot_together(
    {
        "Analysis": ["pet_observed", "potet"],
    },
    # hru_id=84966,  # optional - uses current selection if omitted
    width=1300,
    height=400,
    title="Potential ET",
    renamer={
        "Analysis: pet_observed": "OpenET Potential ET",
        "Analysis: potet": "Modeled Potential ET",
    },
    colors={
        "OpenET Potential ET": "black",
        "Modeled Potential ET": "orange",
    },
)


# Display it
plot_potet

The modeled and "observed" potential ET are indeed very similar in overall magnitide and seasonal variation but als with respect to many individual fluctuations at shorter scales. We'll produce a similar but more detailed plot for actual ET.

In [ ]:
plot_actet = comparer.hru_plot_together(
    {
        "NHM": ["hru_actet"],
        "OL": ["hru_actet"],
        "Analysis": [
            "aet_observed",
            "hru_actet",
        ],
    },
    # hru_id=84966,  # optional - uses current selection if omitted
    title="Actual ET",
    width=1300,
    height=400,
    renamer={
        "NHM: hru_actet": "NHM: Actual ET",
        "OL: hru_actet": "OL: Actual ET",
        "Analysis: hru_actet": "Analysis: Actual ET",
        "Analysis: aet_observed": "OpenET Actual ET",
    },
    colors={
        "OpenET Actual ET": "black",
        "NHM: Actual ET": run_colors["NHM"],
        "OL: Actual ET": run_colors["OL"],
        "Analysis: Actual ET": run_colors["Analysis"],
    },
)


# Display it
plot_actet

The plot shows that while the observed actual ET analysis dramatically imparts realism to our modeled results, the magnitude of the error reduction in the growing season is only roughly half the total error. 

ARE THERE IRRIGATION RATE LIMITS IMPOSED IN THE CODE?

We can include additional variables in the same plot (if they are in the same units). If we want to set the context for modeled actual ET in the broader conted of observed potential Et and available precipitation, we can easily add those variables. 

In [ ]:
# Plot different variables from different runs
plot_actet_2 = comparer.hru_plot_together(
    {
        "NHM": ["hru_actet"],
        "OL": ["hru_actet"],
        "Analysis": [
            "aet_observed",
            "pet_observed",
            "hru_ppt",
            "hru_actet",
        ],
    },
    # hru_id=84966,  # optional - uses current selection if omitted
    title="Evapotranspiration",
    width=1100,
    height=400,
    renamer={
        "NHM: hru_actet": "NHM: Actual ET",
        "OL: hru_actet": "OL: Actual ET",
        "Analysis: hru_actet": "Analysis: Actual ET",
        "Analysis: hru_ppt": "Modeled Precipitation",
        "Analysis: aet_observed": "OpenET Actual ET",
        "Analysis: pet_observed": "OpenET Potential ET",
    },
    colors={
        "Modeled Precipitation": "darkblue",
        "OpenET Potential ET": "gray",
        "OpenET Actual ET": "black",
        "NHM: Actual ET": run_colors["NHM"],
        "OL: Actual ET": run_colors["OL"],
        "Analysis: Actual ET": run_colors["Analysis"],
    },
)
# Display it
plot_actet_2